In [4]:
import os
from dotenv import load_dotenv
load_dotenv()
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

In [5]:
#Import libraries
from langchain_anthropic  import ChatAnthropic
from crewai import Agent, Process, Crew, Task
from IPython.display import display, Markdown
from crewai.knowledge.source.string_knowledge_source import StringKnowledgeSource
from crewai.knowledge.source.pdf_knowledge_source import PDFKnowledgeSource
from crewai.knowledge.knowledge_config import KnowledgeConfig
import os

### Get the data
* Interviewer
* company
* job position
* job description

In [6]:
# Store the data
interviewer = input("Enter the name of the interviewer: ")
company = input("Enter the name of the company: ")
job_position = input("Enter the job position: ")
job_description = input("Enter the job description: ")

### Interviewer AI Agent

In [7]:
from crewai import LLM
llm = LLM(
    model="anthropic/claude-sonnet-4-20250514",
    api_key=os.getenv("ANTHROPIC_API_KEY"), temperature=0.7, max_tokens=3000
)

In [8]:
interviewer_agent = Agent(
    role="Technical Interviewer",
    goal=(
        f"Conduct an interview for the {job_position} role at {company}, "
        "ask relevant questions, and assess whether the candidate is a strong fit."
    ),
    backstory=(
        f"You are {interviewer}, an experienced interviewer at {company}. "
        f"You are hiring for {job_position}. "
        f"Job description: {job_description}"
    ),
    llm=llm,
    verbose=True
)

In [ ]:
# Define the interview prep task
interview_prep_task = Task(
    description=f"""Interview the user for the job position of {job_position} with the following job description: {job_description}""",
    expected_output=f"Ask only one question to the user that is relevant for the job position {job_position} and that has not been asked before.",
    agent=interviewer_agent)

### AI Coach

In [9]:
#  Build the AI interview coach agent
coach_agent = Agent(
    role="AI Interview Coach",
    goal=f"Coach the user to prepare for an interview for the {job_position} role at {company}. by grading the user's answer.",
    backstory="You are an expert on technical job interviews.",
    llm=llm)

In [ ]:
# Define the coaching task
coaching_task = Task(
    description = f"""Give feedback to the user on its answer""",
    expected_output = f"""Give feedback with bullet points on what was good and back and how to get to the next level.""",
    agent = coach_agent,
    context=[interview_prep_task]
)


### AI Crew

CrewAI knowledge/RAG needs an embedding model. If we do not provide one, the installed CrewAI defaults to ChromaDB’s OpenAI embedding function, which looks for OPENAI_API_KEY.

In [10]:
# Empty list of questions
interview_questions=[]

In [11]:
embedder = {
    "provider": "onnx",
    "config": {}
}

interview_content = " ".join(interview_questions) or "No interview questions have been generated yet."
string_source = StringKnowledgeSource(content=interview_content)

interview_prep_task = Task(
    description=f"""
    Interview the user for the job position of {job_position}
    with the following job description: {job_description}.
    Do not ask a question that has already been asked.
    """,
    expected_output=f"""
    Ask only one question to the user that is relevant for the job position {job_position}.
    """,
    agent=interviewer_agent
)

coaching_task = Task(
    description="Give feedback after the candidate answer is collected.",
    expected_output="""
    Give feedback with bullet points:
    - What was good
    - What was weak or missing
    - How to improve the answer
    - A stronger sample answer
    """,
    agent=coach_agent,
    context=[interview_prep_task]
)

def collect_candidate_answer(task_output):
    question = task_output.raw

    print("\nInterview question:")
    print(question)

    candidate_answer = input("\nYour answer: ")

    interview_questions.append(question)

    coaching_task.description = f"""
    The interviewer asked this question:

    {question}

    The candidate answered:

    {candidate_answer}

    Give feedback to the candidate on their answer.
    """

interview_prep_task.callback = collect_candidate_answer

crew = Crew(
    agents=[interviewer_agent, coach_agent],
    tasks=[interview_prep_task, coaching_task],
    verbose=True,
    process=Process.sequential,
    knowledge_sources=[string_source],
    embedder=embedder
)

result = await crew.kickoff_async()
display(Markdown(result.raw))

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: e5c38a78-7d65-49f4-82a2-0ab3e7052d1f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      Interview the user for the job position of Agent Dev                                                       │
│      with the following job description: In addition to contributing your expertise by designing agentic        │
│  solutions for our clients, you will support their teams as they evolve their software development practices    │
│  through AI and agentic systems, helping them understand, adopt, and master these new approaches.  You will     │
│  therefore act simultaneously as an agentic expert, a developer, and a technical coach, guiding teams toward    │
│  modern, AI‑powered development practices.  Alongside building agent systems capable of orchestrating complex   │
│  processes — from understanding business needs all the way to generating code — you will work closely with      │
│  client teams so they can take ownership of these new methods and integrate them sustainably into their way of  │
│  working.  The goal is ambitious: to significantly increase development velocity for our clients and transform  │
│  how organizations deliver software, all within a highly regulated environment..                                │
│      Do not ask a question that has already been asked.                                                         │
│                                                                                                                 │
│  ID: 8fbecb67-f935-4cd2-ad09-f19efd63eff4                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔍 Knowledge Retrieval ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Knowledge Retrieval Started                                                                                    │
│  Status: Retrieving...                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 📚 Knowledge Retrieved ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Search Query:                                                                                                  │
│  Agent Dev interview questions agentic solutions software development AI systems technical coaching client      │
│  teams development velocity regulated environment                                                               │
│  Knowledge Retrieved:                                                                                           │
│  Additional Information: No interview questions have been generated yet.                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Interviewer                                                                                   │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Interview the user for the job position of Agent Dev                                                       │
│      with the following job description: In addition to contributing your expertise by designing agentic        │
│  solutions for our clients, you will support their teams as they evolve their software development practices    │
│  through AI and agentic systems, helping them understand, adopt, and master these new approaches.  You will     │
│  therefore act simultaneously as an agentic expert, a developer, and a technical coach, guiding teams toward    │
│  modern, AI‑powered development practices.  Alongside building agent systems capable of orchestrating complex   │
│  processes — from understanding business needs all the way to generating code — you will work closely with      │
│  client teams so they can take ownership of these new methods and integrate them sustainably into their way of  │
│  working.  The goal is ambitious: to significantly increase development velocity for our clients and transform  │
│  how organizations deliver software, all within a highly regulated environment..                                │
│      Do not ask a question that has already been asked.                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Interview Coach                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## Feedback on Your Answer                                                                                     │
│                                                                                                                 │
│  ### What was good:                                                                                             │
│  - You mentioned a specific technology (OCR engine) and context (Blue Prism RPA)                                │
│  - You acknowledged real-world challenges (secure banking environment, business integration)                    │
│  - You demonstrated experience with automation systems                                                          │
│                                                                                                                 │
│  ### What was weak or missing:                                                                                  │
│  - **Misalignment with the question**: The interviewer asked about AI agents and autonomous systems, but you    │
│  focused on OCR and RPA, which are different technologies                                                       │
│  - **Lack of agentic system experience**: No mention of AI agents, multi-agent systems, or autonomous           │
│  decision-making capabilities                                                                                   │
│  - **Missing orchestration details**: No discussion of complex, multi-step process orchestration or             │
│  requirements-to-code generation                                                                                │
│  - **Insufficient technical depth**: Limited explanation of system architecture, design patterns, or AI/ML      │
│  components                                                                                                     │
│  - **Vague challenge resolution**: You mentioned challenges but didn't explain how you specifically overcame    │
│  them                                                                                                           │
│  - **No connection to modern AI**: Missing references to LLMs, agent frameworks, or current AI technologies     │
│                                                                                                                 │
│  ### How to improve the answer:                                                                                 │
│  - **Reframe your experience**: Connect your automation work to agentic concepts if possible                    │
│  - **Be more specific**: Provide concrete examples of multi-step processes you've automated                     │
│  - **Show technical depth**: Discuss system architecture, decision-making logic, and integration patterns       │
│  - **Address the full question**: Cover requirements gathering, process orchestration, and code generation      │
│  aspects                                                                                                        │
│  - **Demonstrate problem-solving**: Give specific examples of challenges and your solutions                     │
│  - **Use relevant terminology**: Include AI agent concepts like planning, reasoning, tool usage, and            │
│  autonomous execution                                                                                           │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│      The interviewer asked this question:                                                                       │
│                                                                                                                 │
│      Hello! Thank you for your interest in the Agent Dev position at Vooban. I'm excited to learn more about    │
│  your background and experience.                                                                                │
│                                                                                                                 │
│  Let me start with a foundational question:                                                                     │
│                                                                                                                 │
│  **Can you walk me through your experience with AI agents and autonomous systems? Specifically, I'd like to     │
│  understand what types of agentic solutions you've built or worked with, and how you've approached designing    │
│  systems that can orchestrate complex, multi-step processes from requirements gathering through to code         │
│  generation.**                                                                                                  │
│                                                                                                                 │
│  I'm particularly interested in hearing about any challenges you faced in implementing these systems and how    │
│  you overcame them.                                                                                             │
│                                                                                                                 │
│      The candidate answered:                                                                                    │
│                                                                                                                 │
│      I have built an OCR engine for my automation in Blue Prism RPA bots. The challenges was the setting up     │
│  all the infrastructure in a very secure banking account and then to ensure the integration in the system with  │
│  the business sector.                                                                                           │
│                                                                                                                 │
│      Give feedback to the candidate on their answer.                                                            │
│                                                                                                                 │
│  Agent: AI Interview Coach                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Feedback on Your Answer

### What was good:
- You mentioned a specific technology (OCR engine) and context (Blue Prism RPA)
- You acknowledged real-world challenges (secure banking environment, business integration)
- You demonstrated experience with automation systems

### What was weak or missing:
- **Misalignment with the question**: The interviewer asked about AI agents and autonomous systems, but you focused on OCR and RPA, which are different technologies
- **Lack of agentic system experience**: No mention of AI agents, multi-agent systems, or autonomous decision-making capabilities
- **Missing orchestration details**: No discussion of complex, multi-step process orchestration or requirements-to-code generation
- **Insufficient technical depth**: Limited explanation of system architecture, design patterns, or AI/ML components
- **Vague challenge resolution**: You mentioned challenges but didn't explain how you specifically overcame them
- **No connection to modern AI**: Missing references to LLMs, agent frameworks, or current AI technologies

### How to improve the answer:
- **Reframe your experience**: Connect your automation work to agentic concepts if possible
- **Be more specific**: Provide concrete examples of multi-step processes you've automated
- **Show technical depth**: Discuss system architecture, decision-making logic, and integration patterns
- **Address the full question**: Cover requirements gathering, process orchestration, and code generation aspects
- **Demonstrate problem-solving**: Give specific examples of challenges and your solutions
- **Use relevant terminology**: Include AI agent concepts like planning, reasoning, tool usage, and autonomous execution

### A stronger sample answer:

"I've worked extensively with intelligent automation systems that share many characteristics with modern AI agents. In my role developing RPA solutions at [Company], I built a multi-agent system for document processing that combined OCR with decision-making capabilities.

The system I designed could autonomously:
- Analyze incoming documents to determine processing requirements
- Orchestrate multiple specialized bots for different document types
- Make decisions about exception handling and escalation
- Generate custom processing workflows based on document characteristics

For example, I created an intelligent invoice processing agent that would first classify documents, extract relevant data using OCR, validate information against business rules, and then generate appropriate database entries or exception reports.

The main challenge was building the orchestration layer that could coordinate these different components while handling the complex security requirements in our banking environment. I solved this by implementing a state machine architecture with clear decision points and fallback mechanisms.

While this predates current LLM-based agents, the principles of autonomous decision-making, multi-step orchestration, and adaptive workflow generation directly apply to modern agentic systems. I'm excited to bring this foundation to building more sophisticated AI agents using current technologies like tool-calling LLMs and agent frameworks."

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: e5c38a78-7d65-49f4-82a2-0ab3e7052d1f                                                                       │
│  Final Output: ## Feedback on Your Answer                                                                       │
│                                                                                                                 │
│  ### What was good:                                                                                             │
│  - You mentioned a specific technology (OCR engine) and context (Blue Prism RPA)                                │
│  - You acknowledged real-world challenges (secure banking environment, business integration)                    │
│  - You demonstrated experience with automation systems                                                          │
│                                                                                                                 │
│  ### What was weak or missing:                                                                                  │
│  - **Misalignment with the question**: The interviewer asked about AI agents and autonomous systems, but you    │
│  focused on OCR and RPA, which are different technologies                                                       │
│  - **Lack of agentic system experience**: No mention of AI agents, multi-agent systems, or autonomous           │
│  decision-making capabilities                                                                                   │
│  - **Missing orchestration details**: No discussion of complex, multi-step process orchestration or             │
│  requirements-to-code generation                                                                                │
│  - **Insufficient technical depth**: Limited explanation of system architecture, design patterns, or AI/ML      │
│  components                                                                                                     │
│  - **Vague challenge resolution**: You mentioned challenges but didn't explain how you specifically overcame    │
│  them                                                                                                           │
│  - **No connection to modern AI**: Missing references to LLMs, agent frameworks, or current AI technologies     │
│                                                                                                                 │
│  ### How to improve the answer:                                                                                 │
│  - **Reframe your experience**: Connect your automation work to agentic concepts if possible                    │
│  - **Be more specific**: Provide concrete examples of multi-step processes you've automated                     │
│  - **Show technical depth**: Discuss system architecture, decision-making logic, and integration patterns       │
│  - **Address the full question**: Cover requirements gathering, process orchestration, and code generation      │
│  aspects                                                                                                        │
│  - **Demonstrate problem-solving**: Give specific examples of challenges and your solutions                     │
│  - **Use relevant terminology**: Include AI agent concepts like planning, reasoning, tool usage, and            │
│  autonomous execution                                                                                           │
│                                                       

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯